In [ ]:

from pathlib import Path
import gzip, json, math, sys
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Polygon
from matplotlib.ticker import PercentFormatter
from matplotlib.transforms import Bbox
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'code').is_dir() and (p/'data').is_dir())
sys.path.insert(0, str(ROOT/'code/scripts'))
FIG_DIR = Path('/data/chs/09.project_chs/05.protnegdb_project/deliverables/nature_submission_20260823/figures/regenerated')
MAIN_DIR = FIG_DIR / 'main'
SUPP_DIR = FIG_DIR / 'supplementary'
MAIN_DIR.mkdir(parents=True, exist_ok=True)
SUPP_DIR.mkdir(parents=True, exist_ok=True)

COL = {
    'ink':'#17212B', 'muted':'#66727E', 'grid':'#D7DEE5',
    'blue':'#2F6B9A', 'blue_light':'#DCEAF4',
    'green':'#14866D', 'green_light':'#D8EEE8',
    'orange':'#B26D16', 'orange_light':'#F4E7CF',
    'red':'#D55E45', 'red_light':'#F7E1DB',
    'purple':'#6C5AA7', 'purple_light':'#E8E3F4',
    'grey':'#7B8792', 'grey_light':'#E7EAED',
}
COL.update({
    'ink':COL['ink'], 'pdb':COL['blue'], 'pdb_light':COL['blue_light'],
    'pos':COL['green'], 'pos_light':COL['green_light'],
    'lit':COL['orange'], 'lit_light':COL['orange_light'],
    'neg':COL['red'], 'neg_light':COL['red_light'],
    'rand':COL['grey'], 'rand_light':COL['grey_light'],
})

mpl.rcParams.update({
    'font.family':'Arial', 'font.size':7.2, 'axes.titlesize':8.0,
    'axes.titleweight':'normal', 'axes.labelsize':7.2,
    'axes.edgecolor':COL['ink'], 'axes.linewidth':0.7,
    'xtick.labelsize':6.4, 'ytick.labelsize':6.4,
    'legend.fontsize':6.1, 'legend.frameon':False,
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'savefig.facecolor':'white', 'savefig.bbox':'tight',
    'pdf.fonttype':42, 'ps.fonttype':42, 'svg.fonttype':'none',
})

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def panel(ax, label, title):
    ax.text(-0.10, 1.07, label, transform=ax.transAxes, fontsize=11,
            fontweight='normal', va='top', ha='left', clip_on=False)
    ax.set_title(title, loc='left', pad=7, fontsize=7.7)

def clean(ax, grid=None):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    if grid:
        ax.grid(axis=grid, color=COL['grid'], lw=0.5, zorder=0)
        ax.set_axisbelow(True)

def style():
    # The complete style is declared above so every notebook cell is inspectable.
    mpl.rcParams.update({'axes.titlepad':4, 'legend.handlelength':1.5})

def panel_label(ax, label):
    ax.text(-0.10,1.07,label,transform=ax.transAxes,fontsize=11,fontweight='normal',
            va='top',ha='left',clip_on=False)

def panel_title(ax, title):
    ax.set_title(title,loc='left',pad=7,fontsize=7.7)

def clean_axis(ax, grid=None):
    clean(ax,grid)

def ecdf(values):
    values=np.sort(np.asarray(values,float)[np.isfinite(values)])
    return values,np.arange(1,len(values)+1,dtype=float)/len(values)

def point_ci(ax, x, lo, hi, y, color, marker='o', size=34):
    ax.errorbar(x, y, xerr=[[x-lo],[hi-x]], fmt='none', ecolor=COL['ink'],
                elinewidth=0.85, capsize=2, zorder=2)
    ax.scatter([x],[y], s=size, marker=marker, color=color,
               edgecolor='white', linewidth=0.6, zorder=3)

def violin_summary(ax, arrays, labels, colors, orientation='horizontal', widths=0.82):
    positions = np.arange(1, len(arrays)+1)
    vp = ax.violinplot(arrays, positions=positions, vert=(orientation!='horizontal'),
                       widths=widths, showmeans=False, showmedians=False, showextrema=False)
    for body, color in zip(vp['bodies'], colors):
        body.set_facecolor(color); body.set_edgecolor('none'); body.set_alpha(0.55)
    for pos, values, color in zip(positions, arrays, colors):
        q1, med, q3 = np.quantile(values, [0.25,0.5,0.75])
        if orientation == 'horizontal':
            ax.plot([q1,q3],[pos,pos], color=COL['ink'], lw=2.1, solid_capstyle='round')
            ax.scatter([med],[pos], s=25, color=color, edgecolor='white', lw=0.5, zorder=4)
        else:
            ax.plot([pos,pos],[q1,q3], color=COL['ink'], lw=2.1, solid_capstyle='round')
            ax.scatter([pos],[med], s=25, color=color, edgecolor='white', lw=0.5, zorder=4)
    if orientation == 'horizontal':
        ax.set_yticks(positions); ax.set_yticklabels(labels)
    else:
        ax.set_xticks(positions); ax.set_xticklabels(labels)
    return positions

def save_all(fig, directory, stem):
    for suffix in ('pdf','png','svg'):
        fig.savefig(Path(directory)/f'{stem}.{suffix}', dpi=400)
    plt.close(fig)

def export_panel(fig, axes, directory, stem, extra_artists=(), pad=1.06):
    """Crop one or several axes from an assembled figure for manual composition."""
    directory=Path(directory); directory.mkdir(parents=True,exist_ok=True)
    fig.canvas.draw(); renderer=fig.canvas.get_renderer()
    boxes=[ax.get_tightbbox(renderer) for ax in axes]
    boxes.extend(artist.get_window_extent(renderer) for artist in extra_artists)
    bbox=Bbox.union(boxes).expanded(pad,pad).transformed(fig.dpi_scale_trans.inverted())
    for suffix in ('pdf','png','svg'):
        fig.savefig(directory/f'{stem}.{suffix}',dpi=400,bbox_inches=bbox)


In [ ]:

from Bio.PDB.MMCIF2Dict import MMCIF2Dict

master = pd.read_csv(ROOT/'data/interim/figure12_master_v1/figure2_triplet_master_v1.tsv', sep='\t')
core = master.loc[master.core_flag.eq(1)].copy()
ledger = read_json(ROOT/'data/interim/figure12_master_v1/ledger_v1.json')
pairs = pd.read_csv(ROOT/'data/interim/structure_triplet_scoring_v1/scoring_pairs_v1.tsv', sep='\t')
pair_scores = pd.read_csv(ROOT/'data/interim/structure_triplet_scoring_v1/plminteract_scores_v1.csv')
assert len(pairs) == len(pair_scores)
pairs = pairs.assign(score=pair_scores.score.to_numpy(float))
mint = read_json(ROOT/'data/interim/structure_core_mint_full_v1/mint_full_core_eval_v1.json')
sprint_all = read_json(ROOT/'data/interim/structure_triplet_scoring_v1/core_zero_shot_eval_v1/structure_core_zero_shot_eval_v1.json')

def macro_rank(df, ad, ac, mask=None, n_boot=2000, seed=20260822):
    use = df if mask is None else df.loc[mask]
    use = use.dropna(subset=[ad,ac]).copy()
    use['win'] = np.where(use[ad]>use[ac],1.0,np.where(use[ad]<use[ac],0.0,0.5))
    units = use.groupby('ac_id', sort=False).win.mean().to_numpy(float)
    rng = np.random.default_rng(seed)
    boot = np.array([rng.choice(units, len(units), replace=True).mean() for _ in range(n_boot)])
    return dict(p=float(units.mean()), lo=float(np.quantile(boot,.025)), hi=float(np.quantile(boot,.975)),
                n_triplets=len(use), n_ac=len(units), median_delta=float(np.median(use[ad]-use[ac])))

models = {
    'PLM-Interact': macro_rank(core,'score_AD_PLMInteract','score_AC_PLMInteract'),
    'MINT': dict(p=mint['p_d_gt_c_macro'], lo=mint['ci95_unique_ac_bootstrap'][0],
                 hi=mint['ci95_unique_ac_bootstrap'][1], n_triplets=mint['n_triplets'], n_ac=mint['n_unique_ac']),
    'D-SCRIPT': macro_rank(core,'score_AD_DSCRIPT','score_AC_DSCRIPT',seed=20260823),
    'Topsy-Turvy': macro_rank(core,'score_AD_TopsyTurvy','score_AC_TopsyTurvy',seed=20260824),
}
sprint = sprint_all['models']['sprint']['ranking']['core']
models['SPRINT'] = dict(p=sprint['unique_ac_p_d_gt_c'], lo=sprint['unique_ac_p_d_gt_c_ci95'][0],
                        hi=sprint['unique_ac_p_d_gt_c_ci95'][1], n_triplets=sprint['n_triplets'], n_ac=sprint['n_unique_ac'])

subset_robust = {
    'All core': macro_rank(core,'score_AD_PLMInteract','score_AC_PLMInteract'),
    'Repeated ≥2 PDB': macro_rank(core,'score_AD_PLMInteract','score_AC_PLMInteract',core.replicated_AC.eq(1),seed=7),
}
seq_robust = {}
for label, key in [('SeqID 20–30%','20_30'),('SeqID 30–40%','30_40'),('SeqID 40–50%','40_50'),('SeqID ≥50%','ge50')]:
    seq_robust[label] = macro_rank(core,'score_AD_PLMInteract','score_AC_PLMInteract',core.seqid_CD_bin.eq(key),seed=11+len(seq_robust))

pilot = ledger['original_plminteract']['structure_pilot_three_way']
pilot_rows = [
    ('D > random', pilot['p_d_gt_r'], pilot['p_d_gt_r_ci_low'], pilot['p_d_gt_r_ci_high'], COL['green']),
    ('C > random', pilot['p_c_gt_r'], pilot['p_c_gt_r_ci_low'], pilot['p_c_gt_r_ci_high'], COL['red']),
    ('D > C', pilot['p_d_gt_c'], pilot['p_d_gt_c_ci_low'], pilot['p_d_gt_c_ci_high'], COL['blue']),
]

score_sets = {
    'A–D direct contact': pairs.loc[pairs.role.eq('ad_positive') & pairs.assembly_size_stratum.isin(['3_10','11_24']),'score'].to_numpy(float),
    'A–C assembly noncontact': pairs.loc[pairs.role.eq('ac_negative') & pairs.assembly_size_stratum.isin(['3_10','11_24']),'score'].to_numpy(float),
    'A–R random unlabeled': pairs.loc[pairs.role.eq('random_unlabeled'),'score'].to_numpy(float),
}

PDB_DIR = ROOT/'data/structures'
EXAMPLES = [
    dict(pdb='9GD4', file='9gd4-assembly1.cif.gz', title='Yeast septin heterotetramer', A='B', C='D', D='C'),
    dict(pdb='8S9S', file='8s9s-assembly1.cif.gz', title='Human ER membrane complex', A='F', C='G', D='C'),
    dict(pdb='5ANC', file='5anc-assembly1.cif.gz', title='Ribosome-maturation complex', A='E', C='C', D='J'),
]

def load_chain_ca(path):
    with gzip.open(path, 'rt') as handle:
        mm = MMCIF2Dict(handle)
    def arr(key):
        v = mm[key]; return v if isinstance(v,list) else [v]
    group = arr('_atom_site.group_PDB'); atom = arr('_atom_site.label_atom_id')
    chain = arr('_atom_site.label_asym_id')
    x = arr('_atom_site.Cartn_x'); y = arr('_atom_site.Cartn_y'); z = arr('_atom_site.Cartn_z')
    out = {}
    for g,a,c,xx,yy,zz in zip(group,atom,chain,x,y,z):
        if g == 'ATOM' and a == 'CA': out.setdefault(c,[]).append([float(xx),float(yy),float(zz)])
    return {k:np.asarray(v,float) for k,v in out.items() if len(v)>2}

def project_chains(chains):
    all_xyz = np.vstack(list(chains.values())); centre = all_xyz.mean(0)
    _,_,vt = np.linalg.svd(all_xyz-centre, full_matrices=False); basis = vt[:2].T
    return {k:(v-centre)@basis for k,v in chains.items()}

def draw_structure(ax, ex):
    chains = project_chains(load_chain_ca(PDB_DIR/ex['file']))
    role_col = {ex['A']:COL['blue'], ex['C']:COL['red'], ex['D']:COL['green']}
    for chain, xy in chains.items():
        ax.plot(xy[:,0], xy[:,1], lw=1.7 if chain in role_col else .65,
                color=role_col.get(chain,COL['grey']), alpha=1 if chain in role_col else .20,
                solid_capstyle='round', zorder=2 if chain in role_col else 1)
    for role in ('A','D','C'):
        ch=ex[role]
        if ch in chains:
            xy=chains[ch]; mid=xy[len(xy)//2]
            ax.scatter(*mid,s=95,color=role_col[ch],edgecolor='white',lw=.7,zorder=4)
            ax.text(*mid,role,color='white',ha='center',va='center',fontsize=6.5,fontweight='normal',zorder=5)
    ax.set_aspect('equal'); ax.axis('off'); ax.set_title(f"{ex['pdb']} | {ex['title']}",fontsize=6.7,pad=2)


In [ ]:

# Edit only this cell when changing the composite proportions.
LAYOUT = {
    'figsize': (7.2, 7.55),
    'height_ratios': [1.00, 1.05, 1.00],
    'width_ratios': [1, 1, 1, 1, 1, 1],
    'left': 0.07, 'right': 0.985, 'bottom': 0.07, 'top': 0.88,
    'wspace': 0.95, 'hspace': 0.55,
}

# (row, start column, end column); six equal columns make ratios intuitive.
PANEL_SPANS = {
    'a': (0, 0, 6),       # full-width structural examples
    'c': (1, 0, 3),       # 3/6 width
    'b': (1, 3, 6),       # 3/6 width
    'd': (2, 0, 2),       # 2/6 width
    'e': (2, 2, 4),       # 2/6 width
}

EXPORT_INDIVIDUAL_PANELS = True
PANEL_DIR = MAIN_DIR / 'Figure2_panels'


In [ ]:
# Figure 2b — edit this cell independently.
# One cell: create the small canvas, draw, and write the image files.

def draw_panel_c(ax):
    ax.text(-.14,1.10,'c',transform=ax.transAxes,fontsize=11,fontweight='normal',va='top',ha='left',clip_on=False)
    ax.set_title('PLM-Interact score distributions',loc='left',pad=7,fontsize=7.7)
    curves = [
        ('A–D direct contact', score_sets['A–D direct contact'], COL['green']),
        ('A–C assembly noncontact', score_sets['A–C assembly noncontact'], COL['red']),
        ('A–R random unlabeled', score_sets['A–R random unlabeled'], COL['grey']),
    ]
    for label,values,color in curves:
        xx,yy=ecdf(values)
        ax.plot(xx,yy,color=color,lw=1.55,
                label=f'{label}\nn={len(values):,}; median={np.median(values):.4f}')
    ax.set_xlabel('PPI score')
    ax.set_ylabel('Cumulative fraction')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    clean(ax,'both')
    ax.legend(loc='upper left',handlelength=2.0,labelspacing=.55)

PANEL_DIR.mkdir(parents=True, exist_ok=True)
fig_b, ax_c_small = plt.subplots(figsize=(4, 3.2))
fig_b.subplots_adjust(left=0.16, right=0.97, bottom=0.18, top=0.82)
draw_panel_c(ax_c_small)
stem_b = 'Figure2b'
for suffix in ('pdf','png','svg'):
    fig_b.savefig(PANEL_DIR/f'{stem_b}.{suffix}', dpi=400)
plt.close(fig_b)
display(Image(filename=str(PANEL_DIR/f'{stem_b}.png')))



In [ ]:
PANEL_DIR


In [ ]:
plt


In [ ]:

fig = plt.figure(figsize=LAYOUT['figsize'])
gs = fig.add_gridspec(3,6, height_ratios=LAYOUT['height_ratios'],
                      width_ratios=LAYOUT['width_ratios'],
                      left=LAYOUT['left'],right=LAYOUT['right'],
                      bottom=LAYOUT['bottom'],top=LAYOUT['top'],
                      wspace=LAYOUT['wspace'],hspace=LAYOUT['hspace'])
fig.text(.055,.965,'Figure 2 | Sequence predictors recognize context but poorly resolve direct partners',fontsize=9.8,fontweight='normal',va='top')
fig.text(.055,.938,'Structure examples • score distributions • paired partner ranking • prespecified robustness',fontsize=6.9,color=COL['muted'],va='top')


In [ ]:

_r,_c0,_c1=PANEL_SPANS['a']
top = gs[_r,_c0:_c1].subgridspec(1,3,wspace=.12)
top_axes=[]
for i,ex in enumerate(EXAMPLES):
    ax=fig.add_subplot(top[0,i]); top_axes.append(ax); draw_structure(ax,ex)
a_label=fig.text(.055,.865,'a',fontsize=11,fontweight='normal',va='top')
a_title=fig.text(.078,.865,'Representative assembly contexts',fontsize=7.7,fontweight='normal',va='top')


In [ ]:

_r,_c0,_c1=PANEL_SPANS['c']
ax_c=fig.add_subplot(gs[_r,_c0:_c1])
draw_panel_c(ax_c)


In [ ]:
ax_c


In [ ]:

_r,_c0,_c1=PANEL_SPANS['b']
ax_b=fig.add_subplot(gs[_r,_c0:_c1])
panel(ax_b,'b','Five-model direct-partner ranking')
order=['PLM-Interact','MINT','D-SCRIPT','Topsy-Turvy','SPRINT']
yy=np.arange(len(order))[::-1]
cols=[COL['blue'],COL['green'],COL['purple'],COL['orange'],COL['grey']]
for y,m,c in zip(yy,order,cols):
    b=models[m]
    point_ci(ax_b,b['p'],b['lo'],b['hi'],y,c)
    ax_b.text(b['hi']+.008,y,f"{100*b['p']:.1f}%",va='center',fontsize=5.7)
ax_b.axvline(.5,color=COL['grey'],lw=.8,ls=(0,(3,2)))
ax_b.set_yticks(yy); ax_b.set_yticklabels(order)
ax_b.set_xlabel('P(D>C)'); ax_b.set_xlim(.43,.66)
ax_b.xaxis.set_major_formatter(PercentFormatter(1)); clean(ax_b,'x')


In [ ]:

_r,_c0,_c1=PANEL_SPANS['d']
ax_d=fig.add_subplot(gs[_r,_c0:_c1])
panel(ax_d,'d','Same-anchor random control')
yy=np.arange(3)[::-1]
for y,(lab,p,lo,hi,c) in zip(yy,pilot_rows):
    point_ci(ax_d,p,lo,hi,y,c,size=42)
    ax_d.text(hi+.018,y,f'{100*p:.1f}%',va='center',fontsize=6,fontweight='normal')
ax_d.axvline(.5,color=COL['grey'],lw=.8,ls=(0,(3,2)))
ax_d.set_yticks(yy); ax_d.set_yticklabels([x[0] for x in pilot_rows])
ax_d.set_xlim(.40,1.03); ax_d.set_xlabel('Ordering probability')
ax_d.xaxis.set_major_formatter(PercentFormatter(1)); clean(ax_d,'x')
ax_d.text(.02,.96,'Separate set: 300 triplets; 266 anchors',transform=ax_d.transAxes,va='top',fontsize=5.2,color=COL['muted'])


In [ ]:

_r,_c0,_c1=PANEL_SPANS['e']
ax_e=fig.add_subplot(gs[_r,_c0:_c1])
panel(ax_e,'e','Persistent noncontact remains hard')
order=list(subset_robust); yy=np.arange(len(order))[::-1]
for y,name in zip(yy,order):
    b=subset_robust[name]; c=COL['purple'] if 'Repeated' in name else COL['blue']
    point_ci(ax_e,b['p'],b['lo'],b['hi'],y,c,size=35)
    ax_e.text(b['hi']+.006,y,f"{100*b['p']:.1f}%\nn={b['n_triplets']:,}",va='center',fontsize=5.3)
ax_e.axvline(.5,color=COL['grey'],lw=.8,ls=(0,(3,2)))
ax_e.set_yticks(yy); ax_e.set_yticklabels(order)
ax_e.set_xlabel('PLM-Interact P(D>C)'); ax_e.set_xlim(.46,.61)
ax_e.xaxis.set_major_formatter(PercentFormatter(1)); clean(ax_e,'x')


In [ ]:

stem='Figure2_partner_selection_v1'
if EXPORT_INDIVIDUAL_PANELS:
    export_panel(fig,top_axes,PANEL_DIR,'Figure2_panel_a',extra_artists=[a_label,a_title],pad=1.04)
    export_panel(fig,[ax_c],PANEL_DIR,'Figure2_panel_c')
    export_panel(fig,[ax_b],PANEL_DIR,'Figure2_panel_b')
    export_panel(fig,[ax_d],PANEL_DIR,'Figure2_panel_d')
    export_panel(fig,[ax_e],PANEL_DIR,'Figure2_panel_e')
save_all(fig,MAIN_DIR,stem)
display(Image(filename=str(MAIN_DIR/f'{stem}.png')))
